In [2]:
import torch
t2 = torch.arange(12).reshape(3, 4)
print(t2)
for row in t2:
    for col in row:
        print(col)
print(t2.dtype)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor(0)
tensor(1)
tensor(2)
tensor(3)
tensor(4)
tensor(5)
tensor(6)
tensor(7)
tensor(8)
tensor(9)
tensor(10)
tensor(11)
torch.int64


In [3]:
import torch.nn as nn
class Embedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.device = device
        self.dtype = dtype
        self.embeddings = nn.Parameter(torch.empty(self.num_embeddings, self.embedding_dim))
        if device:
            self.embeddings.device = device
        if dtype:
            self.embeddings.dtype = dtype
        self.init_embeddings()

    def init_embeddings(self):
        nn.init.trunc_normal_(self.embeddings, 0, 1, -3, 3)

    def forward(self, x:torch.tensor):
        """x为(batch_size, sequence_length)， 每个元素都对应一个token id，在self.embeddings中找到token_id对应的d_model向量，赋值给这个元素"""
        assert x.dtype == torch.int64
        return self.embeddings[x]

embed = Embedding(6,4)
t1 = torch.arange(6).view(2,3)
print(t1)
print(embed.embeddings)
embed_t1 = embed(t1)
print(embed_t1)
print(embed_t1.shape)

tensor([[0, 1, 2],
        [3, 4, 5]])
Parameter containing:
tensor([[-2.4199, -1.0611,  0.8535,  1.8422],
        [ 0.6217,  1.2886,  0.3605,  0.9610],
        [-1.2595,  0.0628,  0.2037,  0.5255],
        [ 0.2513,  1.9519,  1.2515, -1.7160],
        [ 0.4680, -0.8231,  0.7382, -0.6363],
        [ 1.1898, -1.2267, -0.5257,  0.9913]], requires_grad=True)
tensor([[[-2.4199, -1.0611,  0.8535,  1.8422],
         [ 0.6217,  1.2886,  0.3605,  0.9610],
         [-1.2595,  0.0628,  0.2037,  0.5255]],

        [[ 0.2513,  1.9519,  1.2515, -1.7160],
         [ 0.4680, -0.8231,  0.7382, -0.6363],
         [ 1.1898, -1.2267, -0.5257,  0.9913]]], grad_fn=<IndexBackward0>)
torch.Size([2, 3, 4])


In [4]:
t2 = torch.arange(24).view(2,3,4)
t2 = t2.to(torch.float32)
z = t2.pow(2)
print(z)
y = t2.pow(2).mean(-1,keepdim=True)
y1 = torch.sqrt(y)
y2 = torch.rsqrt(y)
print(y,y1,y2)

tensor([[[  0.,   1.,   4.,   9.],
         [ 16.,  25.,  36.,  49.],
         [ 64.,  81., 100., 121.]],

        [[144., 169., 196., 225.],
         [256., 289., 324., 361.],
         [400., 441., 484., 529.]]])
tensor([[[  3.5000],
         [ 31.5000],
         [ 91.5000]],

        [[183.5000],
         [307.5000],
         [463.5000]]]) tensor([[[ 1.8708],
         [ 5.6125],
         [ 9.5656]],

        [[13.5462],
         [17.5357],
         [21.5291]]]) tensor([[[0.5345],
         [0.1782],
         [0.1045]],

        [[0.0738],
         [0.0570],
         [0.0464]]])


In [5]:
def softmax(x: torch.Tensor, dim: int):
    y = x - x.max(dim=dim, keepdim=True).values
    ex_x = y.exp()
    ex_sum = ex_x.sum(dim=dim, keepdim=True)
    return ex_x / ex_sum
t3 = torch.ones(8)
print(t3)
print(t3.max(dim=0,keepdim=True))
print(softmax(t3,0))

tensor([1., 1., 1., 1., 1., 1., 1., 1.])
torch.return_types.max(
values=tensor([1.]),
indices=tensor([0]))
tensor([0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250])


In [6]:
import regex
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
text = "Hello, where are you from?"
words = [word.group() for word in regex.finditer(PAT, text)]
ids = [list(word.encode('utf-8')) for word in words]
print(ids)

[[72, 101, 108, 108, 111], [44], [32, 119, 104, 101, 114, 101], [32, 97, 114, 101], [32, 121, 111, 117], [32, 102, 114, 111, 109], [63]]


In [11]:
causal_down = torch.tril(torch.ones(5,5,dtype=bool))
causal_up = torch.triu(torch.ones(5,5,dtype=bool),diagonal=1)
print(causal_down, causal_up)

tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]]) tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


In [12]:
vocab_size = 50257
d_model = 1600
d_ff = 4288
n_layers = 48
num_heads = 25
context_length = 1024
each_transformer_layer = 8 *  context_length * d_model**2 + 4 * context_length**2 * d_model
proj_num =2 * d_model * context_length * vocab_size
total_flops = n_layers * each_transformer_layer + proj_num
print(total_flops)

1493437644800


In [8]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]  # 获取学习率。
            for p in group["params"]:
                if p.grad is None:
                    continue
                print(p.shape)
                state = self.state[p]  # 获取与 p 关联的状态。
                t = state.get("t", 0)  # 从状态中获取迭代次数，或 0。
                grad = p.grad.data  # 获取损失对 p 的梯度。
                p.data -= lr / math.sqrt(t + 1) * grad  # 原地更新权重张量。
                state["t"] = t + 1  # 增加迭代次数。

        return loss

weights = torch.nn.Parameter(5 * torch.randn(10, 10))
opt = SGD([weights], lr=1e-1)
print(type(weights))

for t in range(10):
    opt.zero_grad()  # 重置所有可学习参数的梯度。
    loss = (weights**2).mean()  # 计算标量损失值。
    print(loss.cpu().item())
    loss.backward()  # 运行反向传播，计算梯度。
    opt.step()  # 运行优化器步骤。

<class 'torch.nn.parameter.Parameter'>
31.811182022094727
torch.Size([10, 10])
31.684066772460938
torch.Size([10, 10])
31.594514846801758
torch.Size([10, 10])
31.521591186523438
torch.Size([10, 10])
31.458576202392578
torch.Size([10, 10])
31.402326583862305
torch.Size([10, 10])
31.3510684967041
torch.Size([10, 10])
31.303691864013672
torch.Size([10, 10])
31.259435653686523
torch.Size([10, 10])
31.217771530151367
torch.Size([10, 10])
